# ✅ Exercice 4 — Entraînement, validation et évaluation

**Objectif :** entraîner le modèle CNN–LSTM–MLP de l’exercice 3, en utilisant une partie du jeu d’entraînement comme validation, puis évaluer sur le jeu de test.

**✍️ 4.1. Division en validation**

1.	À partir de l’ensemble d’entraînement (Data_Train_PPG.npy et Data_Train_Labels.npy), créez un sous-ensemble de validation (20%) en utilisant `train_test_split` :

        from sklearn.model_selection import train_test_split
        X_train, X_val, X_train_labels, X_val_labels = train_test_split(train_ppg_scaled, train_labels_scaled, test_size=0.2, random_state=42, shuffle=True)

3.	Vérifiez et reportez les dimensions obtenues pour les trois ensembles : entraînement, validation, test.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, X_train_labels, X_val_labels = train_test_split(train_ppg_s, train_labels_s, test_size=0.2, random_state=42, shuffle=True)


**4.2. Entraînement avec EarlyStopping**
Maintenant que l’architecture de notre modèle est définie, vous allez l’entraîner à l’aide des ensembles de données préparés. Le processus d’entraînement consistera à compiler le modèle et à l’ajuster aux données d’entraînement.

**Ajouter une dimension supplémentaire**
Avant l’entraînement, vous devez ajouter une dimension supplémentaire à vos données d’entrée (train, validation et test) afin de correspondre à
la forme attendue par la couche Conv1D

**Pourquoi ajouter une dimension supplémentaire ?**
Les couches Conv1D ne traitent pas simplement un tableau 2D (exemples, longueur) — elles attendent une entrée 3D de la forme :

(nb_exemples, longueur_sequence, nb_canaux)

* nb_exemples = nombre de segments PPG (par ex. 4000).
* longueur_sequence = longueur de chaque segment (1250 points/features).
* nb_canaux = nombre de signaux différents mesurés en parallèle.

Dans notre cas, le PPG est **un signal unique** → donc nb_canaux = 1.

On ajoute cette 3ᵉ dimension avec `tf.expand_dims(data, axis=-1)` pour que Keras comprenne qu’il s’agit bien d’un signal 1 canal (comme une image en niveaux de gris avec 1 canal)


In [ ]:
# Ajout de la 3e dimension pour Conv1D
X_train_exp = tf.expand_dims(X_train, axis=-1)
X_val_exp   = tf.expand_dims(X_val, axis=-1)

**Construire les ensembles d'entraînement et de validation**
En apprentissage profond, on n’entraîne pas le réseau avec tous les exemples à la fois, mais par petits groupes appelés mini-lots (batches)

**Choix du batch size:** En apprentissage profond, on n’entraîne pas le réseau avec tous les exemples à la fois, mais par petits groupes appelés mini-lots (batches).
* Petit batch size → plus de mises à jour des poids, convergence plus rapide, mais calcul plus bruité.
* Grand batch size → entraînement plus stable, mais plus coûteux en mémoire GPU/CPU.

Ici, vous devez définir un batch_size de **16** et l’utiliser pour construire vos ensembles d’entraînement et de validation :

`train_set = tf.data.Dataset.from_tensor_slices((X_train_expanded, X_train_labels)).shuffle(4000).batch(batch_size)`
`val_set = tf.data.Dataset.from_tensor_slices((X_val_expanded, X_val_labels)).batch(batch_size)`

* `tf.data.Dataset.from_tensor_slices(...)` : crée un data set de paires (signal, labels)
* `.shuffle(4000)` : Permet de mélanger complètement les données avant l’entraînement (évites que les signaux soient vus toujours dans le même ordre). On ne shuffle pas le dataset de validation.
*  `.batch(batch_size)` : regroupes les données par batches.

In [ ]:
# Création des ensembles TensorFlow
batch_size = 16
train_set = tf.data.Dataset.from_tensor_slices((X_train_exp, X_train_labels)).shuffle(4000).batch(batch_size)
val_set   = tf.data.Dataset.from_tensor_slices((X_val_exp, X_val_labels)).batch(batch_size)

**Compiler le modèle :** 
Compilez le modèle avec un optimiseur, une fonction de perte adaptée à une tâche de régression et une métrique d’évaluation, en utilisant une taille de
batch appropriée.

* Utilisez l’optimiseur Adam avec un learning_rate de 0.0001.
* Utilisez MSE (Mean Squared Error) comme fonction de perte.
* Utilisez MAE (Mean Absolute Error) comme métrique à suivre pendant l’entraînement.
* Utilisez une taille de batch de 16

-------------------------------------------------------------------------------------------------------------------------------------

* `model = build_cnn_lstm_mlp(...)` : creer un modèle en appelant la fonction personnelle
* `model.compile(...)` : configure comment le modèle va s'entraîner.
* `optimizer=tf.keras.optimizers.Adam(learning_rate=...)` : optimiseur ADAM et le pas d'apprentissage.
* `loss='mse'` : fonction de perte, le modèle essaye de minimiser le MSE.
* `metrics=['mae']` : Métrique (mean absolute error).

In [ ]:
# Compiler le modèle
lr = 0.0001
model = build_cnn_lstm_mlp((1250, 1))
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
    loss='mse',
    metrics=['mae']
)

**Définir les callbacks:** 
Pour éviter le surapprentissage (overfitting) et sauvegarder le meilleur modèle, définissez deux callbacks.

* **EarlyStopping :** surveillez val_loss et arrêtez l’entraînement s’il n’y a pas d’amélioration pendant 20 époques consécutives (patience=20). Assurez-vous de restaurer les meilleurs poids (restore_best_weights=True) afin de ne pas perdre le modèle ayant les meilleures performances.
-------------------------------------------------------------------------------------------------------------

* `monitor='val_loss'` : Observe la validation loss après chaque époque.
* `patience=20` : Attends 20 époques sans amélioration avant d’arrêter l’entraînement.
* `restore_best_weights=True` : après EarlyStopping, Keras recharge automatiquement les poids de l’époque où val_loss était au minimum (meilleur modèle)
* `mode='min'` : val_loss doit diminuer pour être meilleure.

------------------------------------------------------------------------------------------------------------

* **ModelCheckpoint :** sauvegardez le modèle uniquement lorsqu’une nouvelle meilleure valeur de val_loss est atteinte. Nommez le fichier best_model.keras.

--------------------------------------------------------------------------------------------------------------

* `f"best_model.keras"` : le nom du fichier
* `save_best_only=True` : sauvegarde le modèle UNIQUEMENT lorsque val_loss atteint une nouvelle valeur minimale.
* `monitor='val_loss'` : la métrique surveillée.
* `mode='min'` : si la val_loss diminue → on sauvegarde.

In [ ]:
# Définir les callbacks
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    mode='min'
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    f"best_model.keras",
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)

**Entraîner le modèle:** Utilisez la méthode `model.fit()` pour entraîner votre modèle sur le jeu de données d’entraînement.
* Entraînez pendant un maximum de 100 époques.
* Passez le jeu de validation à l’argument validation_data afin de surveiller les performances sur l’ensemble de validation.
* Incluez les callbacks définis dans l’argument callbacks.

----------------------------------------------------------------------------------------------

* `model.fit(...)` : la fonction qui entraîne réellement le modèle et enregistre les résultats dans `history`
* `train_set` : l'ensemble d'entraînemen.
* `validation_data=val_set` : l'ensemble de validation.
* `epochs=100` : nombre maximal d'époques.
* `callbacks=[early_stopping, checkpoint]` : indique à Keras d’arrêter automatiquement l’entraînement lorsqu’il n’y a plus d’amélioration `(early_stopping)` et de sauvegarder le meilleur modèle obtenu selon la validation `(checkpoint)`.
* `history` : contient
    * `history.history['loss']`
    * `history.history['val_loss']`
    * `history.history['mae']`
    * `history.history['val_mae']`

In [ ]:
# Entraîner le modèle
history = model.fit(
    train_set,
    validation_data=val_set,
    epochs=100,
    callbacks=[early_stopping, checkpoint]
)

**4.3. Courbes d’entraînement/validation**

* **Extraction des données :** récupérez la perte d’entraînement via `history.history['loss']` et la perte de validation via `history.history['val_loss']`. Le nombre d’époques correspond à range(1, len(train_loss) + 1).
* **Tracer les courbes :** utilisez `plt.plot()` deux fois — une fois pour la perte d’entraînement (epochs, train_loss) et une fois pour la perte de validation (epochs, val_loss).

In [ ]:
import matplotlib.pyplot as plt

train_loss = history.history['loss']
val_loss   = history.history['val_loss']
epochs     = range(1, len(train_loss) + 1)

plt.figure(figsize=(6,4))
plt.plot(epochs, train_loss, label='Training Loss')
plt.plot(epochs, val_loss, label='Validation Loss')
plt.xlabel('Époques')
plt.ylabel('MSE')
plt.title('Courbes de perte - Entraînement vs Validation')
plt.legend()
plt.grid(True)
plt.show()

**4.4. Évaluation du modèle**
Une fois l’entraînement terminé, le meilleur modèle (sauvegardé grâce au callback ModelCheckpoint) sera évalué sur l’ensemble de test indépendant.
* Après l’entraînement, chargez le fichier `best_model.keras` (meilleur modèle) afin d’utiliser la version la plus performante
* Utilisez `model.evaluate()` sur votre ensemble de test pour obtenir la perte finale et la MAE

**Note:** N’oubliez pas que l’ensemble de test doit avoir la même dimension étendue que celles utilisées pour les ensembles d’entraînement et de validation

--------------------------------------------------------------------------------------------------------

* `best_model = tf.keras.models.load_model('best_model.keras')` : Charge depuis le disque le meilleur modèle sauvegardé par ModelCheckpoint.
* `test_loss, test_mae = best_model.evaluate(X_test_exp, test_labels_s)` : Évalue ce modèle sur le jeu de test, en calculant la perte (MSE) et la MAE.


In [ ]:
# Ajouter une 3e dimension
X_test_exp  = tf.expand_dims(test_ppg_s, axis=-1)

# Le meilleur modèle
best_model = tf.keras.models.load_model('best_model_16_1e-04.keras')
test_loss, test_mae = best_model.evaluate(X_test_exp, test_labels_s)
print(f"Test Loss: {test_loss:.4f}, Test MAE: {test_mae:.4f}")


**Faire des prédictions et dénormaliser**
* utilisez `model.predict()` pour obtenir les valeurs de pression artérielle prédites sur l’ensemble de test complet.
* rappelez-vous que les sorties du modèle sont normalisées. Utilisez l’objet label_scaler pour appliquer `inverse_transform` et revenir aux unités originales de pression artérielle (mmHg).

----------------------------------------------------------------------------------------------
* `pred_scaled = best_model.predict(X_test_exp)` : Fait une prédiction du modèle sur le test set, mais dans l’espace normalisé (car le modèle a été entraîné sur des labels normalisés).
* `pred_mmHg = label_scaler.inverse_transform(pred_scaled)` : Applique l’opération inverse de la normalisation pour revenir aux vraies unités physiologiques (mmHg) pour SBP et DBP


In [ ]:
# Prédiction sur l'ensemble de test
pred_scaled = best_model.predict(X_test_exp)

# Dénormaliser les prédictions
pred_mmHg   = label_scaler.inverse_transform(pred_scaled)

**Calculer les métriques (ME, MAE et SDE)**
1. ME (Erreur Moyenne) : `np.mean(predicted - target)`
2. MAE (Erreur Absolue Moyenne) : `mean_absolute_error(target, predicted)`
3. SDE (Écart-type de l’erreur) : `np.std(predicted - target)`

In [ ]:
from sklearn.metrics import mean_absolute_error

# Erreur Moyenne (ME)
me_sbp = np.mean(pred_mmHg[:, 0] - test_labels[:, 0])
me_dbp = np.mean(pred_mmHg[:, 1] - test_labels[:, 1])

# Erreur Absolue Moyenne (MAE)
mae_sbp = mean_absolute_error(test_labels[:, 0], pred_mmHg[:, 0])
mae_dbp = mean_absolute_error(test_labels[:, 1], pred_mmHg[:, 1])

# Écart-type de l’erreur (SDE)
sde_sbp = np.std(pred_mmHg[:, 0] - test_labels[:, 0])
sde_dbp = np.std(pred_mmHg[:, 1] - test_labels[:, 1])

# Résumé
print("=== Résultats du modèle ===")
print(f"SBP → ME: {me_sbp:.2f} mmHg | MAE: {mae_sbp:.2f} mmHg | SDE: {sde_sbp:.2f} mmHg")
print(f"DBP → ME: {me_dbp:.2f} mmHg | MAE: {mae_dbp:.2f} mmHg | SDE: {sde_dbp:.2f} mmHg")

**4.5. Visualisation des résultats**
Une évaluation complète ne se limite pas aux chiffres : la visualisation des performances du modèle permet de mieux comprendre son comportement.

**Graphiques Estimé vs Référence:** 
pour la PAS (SBP) et la PAD (DBP), créez un nuage de points avec les valeurs de référence (cibles) sur l’axe x et les valeurs prédites sur l’axe
y. Ajoutez une ligne de référence y = x pour illustrer l’accord parfait.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12,6))

# --- SBP ---

x_sbp = test_labels[:,0]
y_sbp = pred_mmHg[:,0]

# Droite de régression (best fit) avec poly1d
coeffs_sbp = np.polyfit(x_sbp, y_sbp, 1)   # [a, b]
poly_sbp   = np.poly1d(coeffs_sbp)
x_sbp_line = np.linspace(x_sbp.min(), x_sbp.max(), 100)

axes[0].scatter(x_sbp, y_sbp, s=3, alpha=0.8, color='tab:red', label='Données SBP')
axes[0].plot([40, 220], [40, 220], 'k--', label='y = x (accord parfait)')
axes[0].plot(x_sbp_line, poly_sbp(x_sbp_line),
             color='darkred', linewidth=2,
             label=f'Best fit : y={coeffs_sbp[0]:.2f}x+{coeffs_sbp[1]:.1f}')

axes[0].set_title('SBP : Estimée vs Référence')
axes[0].set_xlabel('Référence (mmHg)')
axes[0].set_ylabel('Prédiction (mmHg)')
axes[0].grid(True)
axes[0].axis('equal')
axes[0].legend(loc='upper left')

# --- DBP ---

x_dbp = test_labels[:,1]
y_dbp = pred_mmHg[:,1]

coeffs_dbp = np.polyfit(x_dbp, y_dbp, 1)
poly_dbp   = np.poly1d(coeffs_dbp)
x_dbp_line = np.linspace(x_dbp.min(), x_dbp.max(), 100)

axes[1].scatter(x_dbp, y_dbp, s=3, alpha=0.8, color='tab:blue', label='Données DBP')
axes[1].plot([20, 140], [20, 140], 'k--', label='y = x (accord parfait)')
axes[1].plot(x_dbp_line, poly_dbp(x_dbp_line),
             color='navy', linewidth=2,
             label=f'Best fit : y={coeffs_dbp[0]:.2f}x+{coeffs_dbp[1]:.1f}')

axes[1].set_title('DBP : Estimée vs Référence')
axes[1].set_xlabel('Référence (mmHg)')
axes[1].set_ylabel('Prédiction (mmHg)')
axes[1].grid(True)
axes[1].axis('equal')
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.show()



**Diagrammes de Bland–Altman :** pour la PAS et la PAD, réalisez un graphique de Bland–Altman.

* L’axe x doit représenter la moyenne des valeurs prédites et cibles : (predicted + target) / 2.
* L’axe y doit représenter la différence entre valeurs prédites et cibles : (predicted - target).
* Tracez des lignes horizontales représentant :
  1. La différence moyenne.
  2. Les limites d’accord à 95 % : différence moyenne ± 1.96 × écart-type des différences

In [ ]:
# === SBP ===
mean_sbp = (pred_mmHg[:, 0] + test_labels[:, 0]) / 2
diff_sbp = pred_mmHg[:, 0] - test_labels[:, 0]
md_sbp = np.mean(diff_sbp)
sd_sbp = np.std(diff_sbp)

plt.figure(figsize=(7,5))
plt.scatter(mean_sbp, diff_sbp, s=3, alpha=0.8, color='tab:red', label='Données SBP')
plt.axhline(md_sbp, color='r', linestyle='--', label=f'Moyenne = {md_sbp:.2f}')
plt.axhline(md_sbp + 1.96*sd_sbp, color='g', linestyle=':', label=f'+1.96 SD = {md_sbp + 1.96*sd_sbp:.2f}')
plt.axhline(md_sbp - 1.96*sd_sbp, color='g', linestyle=':', label=f'-1.96 SD = {md_sbp - 1.96*sd_sbp:.2f}')
plt.title('Diagramme de Bland–Altman – SBP')
plt.xlabel('(Prédite + Référence) / 2  (mmHg)')
plt.ylabel('(Prédite - Référence)  (mmHg)')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()


# === DBP ===
mean_dbp = (pred_mmHg[:, 1] + test_labels[:, 1]) / 2
diff_dbp = pred_mmHg[:, 1] - test_labels[:, 1]
md_dbp = np.mean(diff_dbp)
sd_dbp = np.std(diff_dbp)

plt.figure(figsize=(7,5))
plt.scatter(mean_dbp, diff_dbp, s=3, alpha=0.8, color='tab:blue', label='Données DBP')
plt.axhline(md_dbp, color='r', linestyle='--', label=f'Moyenne = {md_dbp:.2f}')
plt.axhline(md_dbp + 1.96*sd_dbp, color='g', linestyle=':', label=f'+1.96 SD = {md_dbp + 1.96*sd_dbp:.2f}')
plt.axhline(md_dbp - 1.96*sd_dbp, color='g', linestyle=':', label=f'-1.96 SD = {md_dbp - 1.96*sd_dbp:.2f}')
plt.title('Diagramme de Bland–Altman – DBP')
plt.xlabel('(Prédite + Référence) / 2  (mmHg)')
plt.ylabel('(Prédite - Référence)  (mmHg)')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()




# Exercice 5 – Impact du taux d’échantillonnage (20%)
Dans l’exercice précédent, vous avez travaillé avec des signaux PPG échantillonnés à 125 Hz.
Dans cet exercice, vous allez analyser comment la réduction du taux d’échantillonnage affecte la performance du modèle ainsi que le coût de calcul. Vous allez réduire le signal à 20 Hz et 5 Hz.

### 1. Prétraitement et filtrage
* Appliquez un filtre anti-repliement adapté avant le sous-échantillonnage.
* Justifiez vos choix de conception du filtre (fréquence de coupure, ordre, type).

--------------------------------------------------------
def lowpass_filter_batch(X, fs, fc, order=4):
    nyq = fs / 2.0
    Wn = fc / nyq
    b, a = butter(order, Wn, btype='low')
    X_filt = filtfilt(b, a, X, axis=1)
    return X_filt

* `X` : dataset de signaux, de forme typique (n_samples, n_points)
* `fs` : fréquence d’échantillonnage
* `fc` : fréquence de coupure du filtre
* `order` : ordre du filtre
---------------------------------------------------------

def downsample_ppg_dataset(X, fs_orig, fs_new, fc, order=4):
    X = np.asarray(X)
    X_filt = lowpass_filter_batch(X, fs_orig, fc, order=order)
    g = np.gcd(fs_new, fs_orig)
    up = fs_new // g
    down = fs_orig // g
    X_ds = resample_poly(X_filt, up, down, axis=1)
    return X_ds

* `X = np.asarray(X)` : Force X à être un array numpy
* `X_filt = lowpass_filter_batch(X, fs_orig, fc, order=order)` : filtrage anti-repliment
* `X_ds = resample_poly(X_filt, up, down, axis=1)` :  axis = 1 → sous-échantillonne chaque signal sur le temps


In [ ]:
# le filtre pass bas pour un dataset
def lowpass_filter_batch(X, fs, fc, order=4):
    nyq = fs / 2.0
    Wn = fc / nyq
    b, a = butter(order, Wn, btype='low')
    X_filt = filtfilt(b, a, X, axis=1)
    return X_filt

# Fonction pour downsample tout un dataset 
def downsample_ppg_dataset(X, fs_orig, fs_new, fc, order=4):
    X = np.asarray(X)
    # 1) Filtrage anti-repliement
    X_filt = lowpass_filter_batch(X, fs_orig, fc, order=order)

    # 2) Sous-échantillonnage par resample_poly
    g = np.gcd(fs_new, fs_orig)
    up = fs_new // g
    down = fs_orig // g
    X_ds = resample_poly(X_filt, up, down, axis=1)
    return X_ds
